<a href="https://colab.research.google.com/github/khu3086/FastAICourse/blob/main/Lec02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project: Interactive Bear Classifier
This notebook demonstrates a complete machine learning pipeline using `fastai`. Due to search API limitations, we use a sample dataset to simulate the classification of Grizzly, Black, and Teddy bears.

### Note on GitHub Rendering
If you see a `the 'state' key is missing from 'metadata.widgets'` error on GitHub, it is because of how interactive widgets are saved. You can fix this by going to **Edit -> Notebook settings** and toggling **'Include widget state in further saves of this notebook'**.

In [137]:
# This cell ensures that we don't save broken widget state in the metadata if you are pushing to Git
import json
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

## 1. Setup and Dependencies
We begin by installing the `fastbook` library and importing the necessary vision and widget modules from `fastai`.

In [138]:
import sys
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

if 'google.colab' in sys.modules:
    !pip install -Uqq fastbook

from fastai.vision.all import *
from fastai.vision.widgets import *
from fastbook import *
import os, shutil
from pathlib import Path

## 2. Data Gathering (Simulation)
To avoid API rate-limiting issues, we utilize the Oxford-IIIT Pet Dataset as a reliable source to simulate our 'Grizzly', 'Black', and 'Teddy' bear categories.

In [139]:
path = Path('bears')
if path.exists(): shutil.rmtree(path)

print("Downloading and preparing dataset...")
path_data = untar_data(URLs.PETS)
path_data_ims = path_data/'images'
path.mkdir(exist_ok=True)

for bear_type in ['grizzly', 'black', 'teddy']:
    dest = path/bear_type
    dest.mkdir(exist_ok=True)
    sample_ims = get_image_files(path_data_ims)
    for i in range(min(20, len(sample_ims))):
        shutil.copy(sample_ims[i], dest/f'{bear_type}_{i}.jpg')

fns = get_image_files(path)
print(f"Data Prep Complete: {len(fns)} images found.")

Data Prep Complete: 60 images found.


## 3. Data Pipeline
We use the `DataBlock` API to define the transformation pipeline: loading images, assigning labels from folder names, splitting the data, and resizing images to 128x128.

In [140]:
bears = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=Resize(128)
)
dls = bears.dataloaders(path)

## 4. Model Training
We use Transfer Learning with a ResNet18 architecture. `fine_tune` will train the head of the model and then the entire model for a few epochs.

In [141]:
print("Fine-tuning ResNet18...")
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(2)
learn.export()
print("Model trained and exported as export.pkl")

Fine-tuning ResNet18...


epoch,train_loss,valid_loss,error_rate,time
0,nan,1.617345,0.500000,00:01


/usr/local/lib/python3.12/dist-packages/fastprogress/fastprogress.py:93: UserWarning: Your generator is empty.
  warn("Your generator is empty.")
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284ee40>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284f930>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284ee40>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284f930>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284ee40>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(

epoch,train_loss,valid_loss,error_rate,time
0,nan,1.617345,0.500000,00:02
1,nan,1.617345,0.500000,00:01


/usr/local/lib/python3.12/dist-packages/fastprogress/fastprogress.py:93: UserWarning: Your generator is empty.
  warn("Your generator is empty.")
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284ee40>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284f930>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284ee40>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284f930>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284ee40>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(

Model trained and exported as export.pkl


## 5. Interactive Classifier App
This section creates a graphical interface where you can upload an image and get an instant classification prediction.

## 6. Deployment Preparation
To deploy this as a standalone app, we need to create two main files: `app.py` (the Streamlit app) and `requirements.txt`.

In [144]:
%%writefile app.py
import gradio as gr
from fastai.vision.all import *

# Load the model
learn = load_learner('export.pkl')
labels = learn.dls.vocab

def predict(img):
    img = PILImage.create(img)
    pred, pred_idx, probs = learn.predict(img)
    return {labels[i]: float(probs[i]) for i in range(len(labels))}

# Define the Gradio interface
interface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=3),
    title="Bear Classifier",
    description="Upload an image to see if it's a Grizzly, Black, or Teddy bear!"
)

interface.launch()

Overwriting app.py


In [145]:
%%writefile requirements.txt
fastai
gradio
torch
ipython

Overwriting requirements.txt


### Next Steps for Deployment:
1. Download `app.py`, `requirements.txt`, and `export.pkl` from the Colab file browser (left sidebar).
2. Go to [Hugging Face Spaces](https://huggingface.co/spaces) and create a new Space.
3. Select **Streamlit** as the SDK.
4. Upload these three files to your new Space. Your app will build and go live automatically!